# Week 1: End-to-end ML notebook (Titanic)
**How to use:** run one cell at a time (Shift+Enter). Read the note above each cell, run it, look at the output, then do the **Your turn** line before moving on. Goal of the notebook: predict whether a passenger survived (classification).

Run this in Google Colab: colab.research.google.com > Upload > pick this file. Pandas, scikit-learn and xgboost are already installed there.

## Step 1. Imports
These are the tools: pandas for tables, scikit-learn for models and metrics, xgboost for the third model.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Step 2. Load the data
Each row is a passenger. The column `survived` (0 or 1) is the **label**: the answer we want to predict. The other columns are **features**.

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)
df.head()

**Your turn:** write one sentence in your notes: how many rows and columns are there? (run `df.shape`)

In [ ]:
df.shape

## Step 3. Explore the data (EDA)
Check for missing values and see how balanced the label is.

In [ ]:
print(df.isna().sum()[lambda s: s > 0])   # columns with missing values
print()
print(df["survived"].value_counts(normalize=True))   # share who survived vs not

In [ ]:
print(df.groupby("sex")["survived"].mean())
print()
print(df.groupby("pclass")["survived"].mean())

**Your turn:** write what you notice. (Hint: which group survived much more, and what does that tell you about which features will matter?)

## Step 4. Clean the data and build features
Models need numbers with no gaps. We keep a few useful columns, fill missing values, and turn text into numbers.
- `age` missing: fill with the median.
- `embarked` missing: fill with the most common value.
- `sex`: male = 0, female = 1.
- `embarked`: one-hot encode (a separate 0/1 column per port).

In [ ]:
data = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()

data["age"] = data["age"].fillna(data["age"].median())
data["embarked"] = data["embarked"].fillna(data["embarked"].mode()[0])
data["sex"] = data["sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["embarked"], drop_first=True)

data.head()

In [ ]:
data.isna().sum().sum()   # should be 0

## Step 5. Split into train and test
The model learns from the **training set**. The **test set** is held back and only used at the end, to check it works on data it has never seen. `stratify=y` keeps the survived/not-survived ratio the same in both.

In [ ]:
X = data.drop(columns="survived")   # features
y = data["survived"]                # label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Step 6. Train three models and measure them
Same data, three algorithms. For each: `fit` (learn from training data), `predict` (guess on test data), then compute metrics.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, eval_metric="logloss", random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })

results = pd.DataFrame(rows).round(3)
results

## Step 7. Check for overfitting
Compare accuracy on the training set vs the test set. A big gap means the model memorized the training data.

In [ ]:
for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: train={train_acc:.3f}  test={test_acc:.3f}")

**Your turn:** which model has the biggest gap between train and test? Write one sentence on what that means (this is the overfitting question your mentor will ask).

## Step 8. Conclusion (write this yourself)
In 4 to 6 sentences, answer:
1. Which model did best, and by how much?
2. Which metric would you care about most for this problem, and why?
3. Which features seem to matter most? (try `models['Random Forest'].feature_importances_`)
4. What would you try next to improve it?

## Step 9. After this works: refactor (later, not today)
Move the reusable parts into three small files: `preprocessing.py` (cleaning and features), `train.py` (fit models), `evaluate.py` (metrics). Each file does one job (this is the S in SOLID: single responsibility). Ask me and I will show you a starter.